# Week 7 Assignment - Document Question Answering System (RAG)

Name: Deeptesh Mohapatra

Goal: build a simple Retrieval-Augmented Generation (RAG) system that answers questions from a
custom document. Instead of relying on what a language model already knows, the system first
finds the relevant parts of the document and then uses a language model to write an answer based
on those parts. This keeps the answers grounded in the actual document and lets the system answer
questions about private data that the model was never trained on.

The pipeline has these stages (the standard RAG design):

1. Document ingestion - load a PDF and pull out its text
2. Text chunking - split the text into small pieces
3. Embedding - turn each piece into a vector that captures its meaning
4. Vector store - keep the vectors so we can search them
5. Query processing - turn the user question into a vector
6. Retrieval - find the pieces most similar to the question
7. Generation - a language model writes the answer using the retrieved pieces

To show that RAG really works on private data, I use a made-up company handbook. A plain language
model cannot know these facts because the company is fictional, so any correct answer has to come
from the document through retrieval.

## Setup

The system uses three pieces:

- sentence-transformers (all-mpnet-base-v2) to turn text into embedding vectors
- numpy for the vector store and cosine similarity search
- a flan-t5-base language model (through transformers) to generate the answers

The models download once from Hugging Face and are then cached, so the first run of the loading
cells can take a minute.

In [1]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'

import numpy as np
import textwrap

from fpdf import FPDF
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

print('Imports OK')

C:\Users\lenovo\Desktop\Celebal_Internship\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Imports OK


## 1. The document (custom / private data)

RAG is meant for your own documents - notes, a resume, research papers, or a company handbook.
Here I write a short handbook for a fictional company, Zephyr Robotics, and save it as a PDF. The
facts in it are invented, which is exactly the point: a language model on its own cannot know
them, so it is a fair test of whether the retrieval part is doing its job.

In [2]:
document_text = """Zephyr Robotics Company Handbook

Company Overview
Zephyr Robotics is a robotics company that designs and builds warehouse automation robots. It was founded in 2016 by Dr. Amara Okafor and is headquartered in Bristol, United Kingdom. The company's mission is to make warehouse automation affordable for small and medium sized businesses that cannot afford traditional industrial robots. As of 2024 the company employs about 450 people.

Leadership
Dr. Amara Okafor serves as the Chief Executive Officer and is also the founder of the company. The Chief Technology Officer is Rahul Menon, who leads the engineering and research teams. The Chief Operating Officer is Elena Rossi, who joined the company in 2018.

Products
Zephyr Robotics sells three main products. The first is the Zephyr Glide, an autonomous mobile robot that moves pallets around a warehouse. It has a payload capacity of 800 kilograms and a battery life of about 9 hours on a single charge. The Zephyr Glide was launched in 2019. The second product is the Zephyr Sort, a robotic arm designed for sorting parcels. It was launched in 2021 and can sort up to 1200 parcels per hour. The third product is Zephyr Mind, a cloud based software platform that coordinates fleets of robots working together. Zephyr Mind is sold as a monthly subscription.

Offices
The company has three offices. The headquarters is in Bristol, United Kingdom. A second office opened in Berlin, Germany in 2020, and a third office opened in Singapore in 2022. The Singapore office focuses on sales and support for customers in the Asia Pacific region.

Funding
Zephyr Robotics has raised money in several rounds. In 2023 the company raised a Series B round of 60 million US dollars. The round was led by Northwind Ventures, with participation from several existing investors. The money is being used to expand manufacturing and to grow the engineering team.

Employee Policies
Zephyr Robotics offers its employees 28 days of paid annual leave each year, in addition to public holidays. Employees are allowed to work remotely up to 3 days per week. Every employee also receives an annual learning budget of 1500 pounds that can be spent on courses, books, or conferences. New employees go through a two week onboarding program before joining their teams.

Customer Support
Customers can contact support by email at support at zephyrrobotics dot example. Phone support lines are open 24 hours a day, 7 days a week for customers with an active service contract. The company aims to respond to all support requests within four hours.

Sustainability
Zephyr Robotics has committed to becoming carbon neutral by 2030. All of its robots are built using recyclable aluminium frames, and the company runs its offices and factories on renewable electricity where possible. Old robots can be returned to the company for recycling at the end of their life.
"""

from fpdf.enums import XPos, YPos
pdf = FPDF()
pdf.add_page()
pdf.set_font('Helvetica', size=11)
for line in document_text.split('\n'):
    if line.strip() == '':
        pdf.ln(4)                       # blank line between sections
    else:
        pdf.multi_cell(0, 6, line, new_x=XPos.LMARGIN, new_y=YPos.NEXT)
pdf_path = 'zephyr_handbook.pdf'
pdf.output(pdf_path)
print('Saved PDF:', pdf_path)
print('Document length:', len(document_text), 'characters')

Saved PDF: zephyr_handbook.pdf
Document length: 2884 characters


## 2. Document ingestion

Now I read the PDF back with pypdf and pull out the raw text - this is the step that would work
on any real PDF you drop in.

In [3]:
reader = PdfReader(pdf_path)
raw_text = "\n".join(page.extract_text() for page in reader.pages)
raw_text = raw_text.strip()
print('Pages:', len(reader.pages))
print('Extracted characters:', len(raw_text))
print('\nFirst 300 characters of extracted text:\n')
print(raw_text[:300])

Pages: 1
Extracted characters: 2875

First 300 characters of extracted text:

Zephyr Robotics Company Handbook
Company Overview
Zephyr Robotics is a robotics company that designs and builds warehouse automation robots. It was founded
in 2016 by Dr. Amara Okafor and is headquartered in Bristol, United Kingdom. The company's mission is to
make warehouse automation affordable fo


## 3. Text chunking

A whole document is too big to send to the model, and searching over big blocks is imprecise. So
I split the text into small chunks. I chunk by sentence: split the text into sentences, then group
every two sentences together with a one sentence overlap so a fact is not cut in half at a
boundary. Splitting on sentences keeps each fact together, which makes retrieval much more
accurate than splitting on a fixed number of words.

One small detail: a naive sentence splitter breaks on the period in "Dr.", so I protect a few
common abbreviations first so names like "Dr. Amara Okafor" stay in one piece.

In [4]:
import re

def split_sentences(text):
    flat = text.replace('\n', ' ')
    for ab in ['Dr.', 'Mr.', 'Mrs.', 'Ms.', 'Inc.', 'Ltd.', 'U.S.']:
        flat = flat.replace(ab, ab.replace('.', '<DOT>'))     # protect abbreviations
    parts = re.split(r'(?<=[.!?])\s+', flat)
    return [p.replace('<DOT>', '.').strip() for p in parts if p.strip()]

def chunk_sentences(sentences, size=2, overlap=1):
    chunks, i, step = [], 0, max(1, size - overlap)
    while i < len(sentences):
        chunks.append(' '.join(sentences[i:i + size]))
        i += step
    return chunks

sentences = split_sentences(raw_text)
chunks = chunk_sentences(sentences, size=2, overlap=1)
print('Number of sentences:', len(sentences))
print('Number of chunks:', len(chunks))
print('\nExample chunk (chunk 3):\n')
print(textwrap.fill(chunks[3], 90))

Number of sentences: 33
Number of chunks: 33

Example chunk (chunk 3):

As of 2024 the company employs about 450 people. Leadership Dr. Amara Okafor serves as the
Chief Executive Officer and is also the founder of the company.


## 4. Embedding the chunks

I load the sentence-transformers model and turn every chunk into a 768-number vector. Chunks that
mean similar things end up close together in this vector space, which is what makes semantic
search possible. I use all-mpnet-base-v2 here because it retrieves more accurately than the
smaller MiniLM model - for example it correctly tells the Chief Executive Officer apart from the
Chief Operating Officer, which a smaller model got confused about. I normalize the vectors so a
dot product is the same as cosine similarity.

In [5]:
embedder = SentenceTransformer('all-mpnet-base-v2')
chunk_embeddings = embedder.encode(chunks, normalize_embeddings=True, show_progress_bar=False)
print('Embedding matrix shape:', chunk_embeddings.shape, '(chunks x vector size)')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2034.85it/s]

Embedding matrix shape: (33, 768) (chunks x vector size)


## 5. Vector store and retrieval

The set of chunk vectors is my vector store (a small numpy array here; a bigger project would use
a database like FAISS or Chroma). To answer a question I embed the question the same way and keep
the chunks whose vectors are most similar to it (highest cosine similarity).

In [6]:
def retrieve(question, k=3):
    q_vec = embedder.encode([question], normalize_embeddings=True)[0]
    scores = chunk_embeddings @ q_vec          # cosine similarity for every chunk
    top_idx = np.argsort(-scores)[:k]
    return [(chunks[i], float(scores[i])) for i in top_idx]

# quick check
demo_q = 'How many days of paid leave do employees get?'
print('Question:', demo_q, '\n')
for text_chunk, score in retrieve(demo_q, k=3):
    print('score', round(score, 3), '->', textwrap.shorten(text_chunk, 100))

Question: How many days of paid leave do employees get? 

score 0.555 -> The money is being used to expand manufacturing and to grow the engineering team. Employee [...]
score 0.514 -> Employee Policies Zephyr Robotics offers its employees 28 days of paid annual leave each year, [...]
score 0.444 -> Employees are allowed to work remotely up to 3 days per week. Every employee also receives an [...]


## 6. The generator (language model)

I load flan-t5-base. Given a prompt that contains the retrieved context and the question, it
writes a short answer. The prompt tells it to use only the context and to say so if the answer is
not there, which keeps it honest instead of guessing.

In [7]:
tokenizer = AutoTokenizer.from_pretrained('google/flan-t5-base')
generator = AutoModelForSeq2SeqLM.from_pretrained('google/flan-t5-base')
print('Language model loaded:', generator.config.name_or_path)

def generate_answer(question, context):
    prompt = (
        "Use the context to answer the question in a short complete sentence. "
        "If the answer is not in the context, reply exactly: "
        "I could not find that in the document.\n\n"
        "Context:\n" + context + "\n\n"
        "Question: " + question + "\nAnswer:")
    ids = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=512)
    with torch.no_grad():
        out = generator.generate(**ids, max_new_tokens=60)
    return tokenizer.decode(out[0], skip_special_tokens=True)

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

Loading weights:  61%|██████▏   | 173/282 [00:00<00:00, 1713.75it/s]

Loading weights: 100%|██████████| 282/282 [00:00<00:00, 1830.11it/s]


[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Language model loaded: google/flan-t5-base


## 7. Putting it together - the RAG function

This ties the two halves together: retrieve the best chunks, join them into a context, and ask
the language model to answer. It also returns the retrieved chunks so we can see what the answer
was based on.

I add one simple guard: a confidence gate. If even the best matching chunk is not similar enough
to the question (below a score of 0.40), the question is probably not covered by the document, so
the system just says it could not find the answer instead of forcing the model to guess.

In [8]:
MIN_SCORE = 0.40   # if nothing is similar enough, treat the question as not covered

def rag_answer(question, k=4, show=True):
    retrieved = retrieve(question, k)
    top_score = retrieved[0][1]
    if top_score < MIN_SCORE:
        answer = 'I could not find that in the document.'
    else:
        context = "\n".join(c for c, _ in retrieved)
        answer = generate_answer(question, context)
    if show:
        print('Q:', question)
        print('Answer:', answer)
        print('  top score', round(top_score, 3), '| retrieved:')
        for c, s in retrieved[:3]:
            print('   score', round(s, 3), '->', textwrap.shorten(c, 85))
        print()
    return answer

questions = [
    'Who founded Zephyr Robotics?',
    'Who is the CEO of Zephyr Robotics?',
    'What is the payload capacity of the Zephyr Glide?',
    'How many days of paid annual leave do employees get?',
    'When did the Berlin office open?',
    'Who led the Series B funding round?',
    'What is Zephyr Mind?',
    'By what year does the company want to be carbon neutral?',
]
for q in questions:
    rag_answer(q)

Q: Who founded Zephyr Robotics?
Answer: Dr. Amara Okafor
  top score 0.743 | retrieved:
   score 0.743 -> Zephyr Robotics Company Handbook Company Overview Zephyr Robotics is a robotics [...]
   score 0.74 -> The Chief Operating Officer is Elena Rossi, who joined the company in 2018. [...]
   score 0.607 -> Sustainability Zephyr Robotics has committed to becoming carbon neutral by [...]



Q: Who is the CEO of Zephyr Robotics?
Answer: Dr. Amara Okafor
  top score 0.828 | retrieved:
   score 0.828 -> The Chief Operating Officer is Elena Rossi, who joined the company in 2018. [...]
   score 0.688 -> Zephyr Robotics Company Handbook Company Overview Zephyr Robotics is a robotics [...]
   score 0.628 -> Leadership Dr. Amara Okafor serves as the Chief Executive Officer and is also [...]



Q: What is the payload capacity of the Zephyr Glide?
Answer: 800 kilograms
  top score 0.813 | retrieved:
   score 0.813 -> It has a payload capacity of 800 kilograms and a battery life of about 9 hours [...]
   score 0.563 -> The first is the Zephyr Glide, an autonomous mobile robot that moves pallets [...]
   score 0.519 -> Products Zephyr Robotics sells three main products. The first is the Zephyr [...]



Q: How many days of paid annual leave do employees get?
Answer: 28
  top score 0.527 | retrieved:
   score 0.527 -> The money is being used to expand manufacturing and to grow the engineering [...]
   score 0.482 -> Employee Policies Zephyr Robotics offers its employees 28 days of paid annual [...]
   score 0.441 -> Employees are allowed to work remotely up to 3 days per week. Every employee [...]



Q: When did the Berlin office open?
Answer: 2020
  top score 0.586 | retrieved:
   score 0.586 -> A second office opened in Berlin, Germany in 2020, and a third office opened in [...]
   score 0.469 -> The headquarters is in Bristol, United Kingdom. A second office opened in [...]
   score 0.322 -> Offices The company has three offices. The headquarters is in Bristol, United [...]



Q: Who led the Series B funding round?
Answer: Northwind Ventures
  top score 0.813 | retrieved:
   score 0.813 -> In 2023 the company raised a Series B round of 60 million US dollars. The round [...]
   score 0.689 -> The round was led by Northwind Ventures, with participation from several [...]
   score 0.491 -> Funding Zephyr Robotics has raised money in several rounds. In 2023 the company [...]



Q: What is Zephyr Mind?
Answer: a cloud based software platform that coordinates fleets of robots working together
  top score 0.58 | retrieved:
   score 0.58 -> Zephyr Mind is sold as a monthly subscription. Offices The company has three offices.
   score 0.553 -> The third product is Zephyr Mind, a cloud based software platform that [...]
   score 0.526 -> It was launched in 2021 and can sort up to 1200 parcels per hour. The third [...]



Q: By what year does the company want to be carbon neutral?
Answer: 2030
  top score 0.526 | retrieved:
   score 0.526 -> The company aims to respond to all support requests within four hours. [...]
   score 0.454 -> Sustainability Zephyr Robotics has committed to becoming carbon neutral by [...]
   score 0.331 -> In 2023 the company raised a Series B round of 60 million US dollars. The round [...]



## 8. Does retrieval actually help? RAG vs the model alone

To show the value of retrieval, I ask the same question two ways: once with no context (just the
language model on its own) and once through the full RAG pipeline. Because the company is
fictional, the model alone has no way to know the answer, while RAG gets it right from the
document.

In [9]:
def plain_answer(question):
    ids = tokenizer(question, return_tensors='pt', truncation=True, max_length=512)
    with torch.no_grad():
        out = generator.generate(**ids, max_new_tokens=60)
    return tokenizer.decode(out[0], skip_special_tokens=True)

for q in ['Who is the CEO of Zephyr Robotics?',
          'What is the payload capacity of the Zephyr Glide?']:
    print('Q:', q)
    print('  Model alone (no context):', plain_answer(q))
    print('  With RAG               :', rag_answer(q, show=False))
    print()

Q: Who is the CEO of Zephyr Robotics?


  Model alone (no context): Jeremy Peters


  With RAG               : Dr. Amara Okafor

Q: What is the payload capacity of the Zephyr Glide?


  Model alone (no context): 2,000 lb


  With RAG               : 800 kilograms



## 9. Questions the document cannot answer

A grounded system should admit when it does not know. Here I ask two questions that have nothing
to do with the handbook. Because no chunk is similar enough to them, the confidence gate stops the
system before it calls the language model, and it simply says it could not find the answer instead
of making something up.

In [10]:
rag_answer('What is the capital of France?')
rag_answer('How do I bake a chocolate cake?')

Q: What is the capital of France?
Answer: I could not find that in the document.
  top score 0.3 | retrieved:
   score 0.3 -> The headquarters is in Bristol, United Kingdom. A second office opened in [...]
   score 0.228 -> A second office opened in Berlin, Germany in 2020, and a third office opened in [...]
   score 0.169 -> Offices The company has three offices. The headquarters is in Bristol, United [...]

Q: How do I bake a chocolate cake?
Answer: I could not find that in the document.
  top score 0.049 | retrieved:
   score 0.049 -> New employees go through a two week onboarding program before joining their [...]
   score 0.045 -> A second office opened in Berlin, Germany in 2020, and a third office opened in [...]
   score 0.017 -> Customer Support Customers can contact support by email at support at [...]



'I could not find that in the document.'

## Conclusion

What I built and what I found:

1. Created a custom PDF (a fictional company handbook) and read the text back out with pypdf.
2. Split the text into small sentence based chunks so each fact could be found on its own.
3. Embedded the chunks with sentence-transformers and stored the vectors as a simple numpy vector
   store.
4. For each question I embedded the question, retrieved the most similar chunks by cosine
   similarity, and passed them as context to a flan-t5 language model that wrote the answer.
5. The system answered the fact based questions correctly - who founded the company, who the CEO
   is, the robot's payload, the leave policy, the funding round, and so on. Getting the CEO right
   actually needed a stronger embedding model: a smaller one kept confusing the Chief Executive
   Officer with the Chief Operating Officer, which is a good reminder that retrieval quality
   matters as much as the language model.
6. The comparison made the point of RAG clear: on its own the language model could not answer
   questions about the fictional company (it made up a name), but with retrieved context it
   answered them correctly.
7. For questions that were unrelated to the document, the confidence gate caught them - the best
   chunk was not similar enough, so the system said it could not find the answer instead of
   guessing.

### Model selection conclusion

The two model choices fit a small CPU friendly setup. all-mpnet-base-v2 is a good, still fairly
small embedding model that retrieves accurately (it correctly separated the CEO from the COO,
which the smaller MiniLM model got wrong). flan-t5-base is a light instruction tuned model that
follows the "answer from this context" instruction well. For a bigger or more demanding system I
would swap in a larger language model, but the pipeline around it would stay the same.

### What worked

- Sentence based chunking: kept each fact together and made retrieval precise.
- A stronger embedding model: fixed a real retrieval mistake (CEO vs COO).
- Normalized embeddings with cosine similarity: a simple and effective vector search.
- A confidence gate plus a "use only the context" prompt: keeps the answers grounded and stops the
  system from answering questions the document does not cover.

### Limitations

- The small flan-t5-base model handles clear fact questions well, but it can still guess on
  questions that are about the right topic yet not actually answered in the document (for example
  a price that is never stated). A larger language model and a re-ranking step would help here.

### Future enhancements

- Hybrid search that mixes keyword matching with vector search, plus a re-ranking step to order
  the retrieved chunks by relevance.
- A larger or instruction tuned language model for smoother, more reliable answers.
- A real vector database (FAISS, Chroma, or Pinecone) so the system scales to thousands of
  documents, and a small web interface so users can upload their own PDFs and ask questions.